# Week 8 - Wednesday Assignment

**Name:** <your name>  
**Program:** PG Diploma AI-ML and Agentic AI Engineering  
**Date:** 2026-04-16

This notebook covers Sub-steps 1-5:
1. Social media dataset loading + characterization
2. MNIST loading + characterization
3. CNN on MNIST (>=2 conv layers) + first-layer filter visualization
4. Hate speech detector + semantic similarity search
5. Two-stage moderation pipeline + recommendation at 100,000 posts/day

---

## AI usage log (required by policy)
- Prompt(s) used:
- What AI generated:
- What I changed and why:


In [ ]:
# Core imports and global config
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

SOCIAL_DATASET_ID = "cosmos98/twitter-and-reddit-sentimental-analysis-dataset"
MNIST_DATASET_ID = "hojjatk/mnist-dataset"

print("Notebook initialized")

## 0) Download / locate datasets
This cell uses `kagglehub` to download both datasets and tries to locate key files automatically.


In [ ]:
import kagglehub

def download_dataset(dataset_id: str) -> Path:
    """Defensive wrapper around kagglehub download."""
    try:
        path = Path(kagglehub.dataset_download(dataset_id))
        print(f"Downloaded: {dataset_id} -> {path}")
        return path
    except Exception as exc:
        raise RuntimeError(
            f"Failed to download {dataset_id}. Check Kaggle credentials / internet."
        ) from exc


def find_file_by_keywords(root: Path, keywords: list[str], suffixes: tuple[str, ...]) -> Path | None:
    """Find first matching file by keywords and suffix."""
    candidates = [p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in suffixes]
    for file_path in candidates:
        name = file_path.name.lower()
        if all(k in name for k in keywords):
            return file_path
    return candidates[0] if candidates else None

social_root = download_dataset(SOCIAL_DATASET_ID)
mnist_root = download_dataset(MNIST_DATASET_ID)

social_csv_path = find_file_by_keywords(social_root, ["social"], (".csv",))
if social_csv_path is None:
    social_csv_path = find_file_by_keywords(social_root, [], (".csv",))

mnist_csv_path = find_file_by_keywords(mnist_root, ["mnist"], (".csv",))

print("social_csv_path:", social_csv_path)
print("mnist_csv_path:", mnist_csv_path)

if social_csv_path is None:
    raise FileNotFoundError("Could not locate a social media CSV file in downloaded dataset.")

## 1) Sub-step 1: Load and characterize social media dataset


In [ ]:
social_df = pd.read_csv(social_csv_path)
print("Shape:", social_df.shape)
display(social_df.head())
print("\nColumns:", list(social_df.columns))
print("\nMissing values per column:")
display(social_df.isna().sum().sort_values(ascending=False))

# Quick profile for likely fields
for col in ["platform", "language", "topic", "sentiment"]:
    if col in social_df.columns:
        print(f"\nValue counts for {col}:")
        display(social_df[col].value_counts(dropna=False).head(20))

# Identify hate/spam candidate columns dynamically
hate_col = next((c for c in social_df.columns if "hate" in c.lower()), None)
spam_col = next((c for c in social_df.columns if "spam" in c.lower()), None)
text_col = next((c for c in social_df.columns if c.lower() in ["text", "post", "content", "tweet", "message"]), None)

print("\nDetected columns:")
print("hate_col:", hate_col)
print("spam_col:", spam_col)
print("text_col:", text_col)

if hate_col:
    print("\nHate speech class distribution:")
    display(social_df[hate_col].value_counts(dropna=False))
if spam_col:
    print("\nSpam class distribution:")
    display(social_df[spam_col].value_counts(dropna=False))

if hate_col:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=social_df, x=hate_col)
    plt.title("Hate-speech label distribution")
    plt.tight_layout()
    plt.show()

### Observations for Sub-step 1 (fill after running)
- Data quality issues found:
- Class distribution issue and why it matters:
- How this affects evaluation choices in Sub-step 5:


## 2) Sub-step 2: Load and characterize MNIST
This section uses `torchvision.datasets.MNIST` (recommended baseline) even though a Kaggle MNIST CSV is also downloaded.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

transform = transforms.Compose([
    transforms.ToTensor(),
])

train_ds = datasets.MNIST(root=DATA_DIR / "mnist_torch", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=DATA_DIR / "mnist_torch", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))

# Distribution across classes
train_targets = train_ds.targets.numpy()
class_counts = pd.Series(train_targets).value_counts().sort_index()
display(class_counts.to_frame("count"))

# Image properties
sample_img, sample_label = train_ds[0]
print("Sample shape:", sample_img.shape, "label:", sample_label)
print("Pixel range:", float(sample_img.min()), "to", float(sample_img.max()))

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    img, lbl = train_ds[i]
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(str(lbl))
    ax.axis("off")
plt.tight_layout()
plt.show()

### Observations for Sub-step 2 (fill after running)
- Digit distribution:
- Image dimensions:
- Pixel range and normalization/preprocessing decisions:


## 3) Sub-step 3: Build and train CNN on MNIST
Architecture includes at least two convolutional layers.


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def evaluate_accuracy(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(x_batch)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / max(total, 1)


model = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x_batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)
    test_acc = evaluate_accuracy(model, test_loader)
    history.append((epoch, train_loss, test_acc))
    print(f"Epoch {epoch}/{EPOCHS} - loss: {train_loss:.4f} - test_acc: {test_acc:.4f}")

history_df = pd.DataFrame(history, columns=["epoch", "train_loss", "test_acc"])
display(history_df)

In [ ]:
# Visualize first conv layer filters
first_conv_weights = model.features[0].weight.detach().cpu()  # shape: [out_channels, 1, k, k]
num_filters = first_conv_weights.shape[0]

cols = 8
rows = int(np.ceil(num_filters / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.5))
axes = np.array(axes).reshape(-1)

for i in range(len(axes)):
    axes[i].axis("off")
    if i < num_filters:
        kernel = first_conv_weights[i, 0]
        axes[i].imshow(kernel, cmap="gray")
        axes[i].set_title(f"F{i}")

plt.suptitle("First convolution layer filters")
plt.tight_layout()
plt.show()

## 4) Sub-step 4: Hate-speech detector + semantic similarity
This section uses:
- TF-IDF + Logistic Regression classifier for hate speech baseline
- SentenceTransformer embeddings + cosine similarity for semantic retrieval


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

if text_col is None or hate_col is None:
    raise ValueError("Could not auto-detect text and hate columns. Update `text_col` / `hate_col` manually.")

work_df = social_df[[text_col, hate_col]].copy()
work_df = work_df.dropna(subset=[text_col, hate_col])
work_df[text_col] = work_df[text_col].astype(str)

# Convert possible labels to binary safely
if work_df[hate_col].dtype == object:
    lowered = work_df[hate_col].astype(str).str.lower().str.strip()
    mapping = {"yes": 1, "true": 1, "hate": 1, "1": 1, "no": 0, "false": 0, "non-hate": 0, "0": 0}
    work_df["hate_binary"] = lowered.map(mapping)
else:
    work_df["hate_binary"] = work_df[hate_col].astype(int)

work_df = work_df.dropna(subset=["hate_binary"])
work_df["hate_binary"] = work_df["hate_binary"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    work_df[text_col],
    work_df["hate_binary"],
    test_size=0.2,
    random_state=SEED,
    stratify=work_df["hate_binary"],
)

vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Handle class imbalance via class_weight='balanced'
hate_clf = LogisticRegression(max_iter=300, class_weight="balanced")
hate_clf.fit(X_train_tfidf, y_train)

y_pred = hate_clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, digits=4))

# Semantic retrieval using sentence embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_texts = work_df[text_col].tolist()
corpus_embeddings = embedding_model.encode(corpus_texts, show_progress_bar=False)

hate_examples = work_df[work_df["hate_binary"] == 1][text_col].head(1).tolist()
if hate_examples:
    query = hate_examples[0]
    query_emb = embedding_model.encode([query], show_progress_bar=False)
    scores = cosine_similarity(query_emb, corpus_embeddings)[0]

    top_k = 10
    top_idx = np.argsort(scores)[-top_k:][::-1]
    retrieval_df = pd.DataFrame({
        "text": [corpus_texts[i] for i in top_idx],
        "similarity": [scores[i] for i in top_idx],
        "hate_binary": [work_df.iloc[i]["hate_binary"] for i in top_idx],
    })

    print("Query:\n", query)
    display(retrieval_df)
else:
    print("No positive hate examples found for retrieval demo.")

## 5) Sub-step 5: Two-stage moderation pipeline
Stage 1: classifier flags likely hate speech.  
Stage 2: embedding similarity retrieves semantically similar posts missed by Stage 1.


In [ ]:
# Stage 1 predictions on full corpus
X_all_tfidf = vectorizer.transform(work_df[text_col])
work_df = work_df.reset_index(drop=True)
work_df["stage1_pred"] = hate_clf.predict(X_all_tfidf)

# Stage 2: retrieval for each stage-1 positive seed
SIM_THRESHOLD = 0.65
MAX_SEEDS = 25  # avoid very heavy O(N^2) retrieval in notebook runtime

seed_indices = work_df.index[work_df["stage1_pred"] == 1].tolist()[:MAX_SEEDS]
stage2_added = set()

for seed_idx in seed_indices:
    q_emb = corpus_embeddings[seed_idx : seed_idx + 1]
    sims = cosine_similarity(q_emb, corpus_embeddings)[0]
    similar_idx = np.where(sims >= SIM_THRESHOLD)[0]
    for idx in similar_idx:
        if work_df.loc[idx, "stage1_pred"] == 0:
            stage2_added.add(int(idx))

work_df["stage2_added"] = 0
if stage2_added:
    work_df.loc[list(stage2_added), "stage2_added"] = 1

work_df["final_flag"] = ((work_df["stage1_pred"] == 1) | (work_df["stage2_added"] == 1)).astype(int)

# Evaluate how many harmful posts stage 2 recovers
true_hate = work_df["hate_binary"] == 1
stage1_hits = ((work_df["stage1_pred"] == 1) & true_hate).sum()
final_hits = ((work_df["final_flag"] == 1) & true_hate).sum()
additional_hits = final_hits - stage1_hits

precision, recall, f1, _ = precision_recall_fscore_support(
    work_df["hate_binary"], work_df["final_flag"], average="binary", zero_division=0
)

print("Stage 1 true harmful captured:", int(stage1_hits))
print("Final pipeline true harmful captured:", int(final_hits))
print("Additional harmful surfaced by Stage 2:", int(additional_hits))
print(f"Pipeline precision={precision:.4f}, recall={recall:.4f}, f1={f1:.4f}")

# Daily estimate for 100,000 posts/day
flag_rate = work_df["final_flag"].mean()
estimated_daily_reviews = int(round(100_000 * flag_rate))
print("Estimated daily review volume at 100,000 posts/day:", estimated_daily_reviews)

display(work_df[[text_col, "hate_binary", "stage1_pred", "stage2_added", "final_flag"]].head(20))

## Recommendation (fill after running)
- Most appropriate metric for Meera's Trust & Safety goal:
- Why this metric matters more than alternatives:
- Additional harmful posts surfaced by Stage 2:
- Estimated daily moderation workload at 100k posts/day:
- Final recommendation:
